# Ch.6 — RNNs / LSTMs / GRUs
**Track:** ML from Scratch · Synthetic monthly housing price index (time-series forecasting)

## Core Idea

Dense networks and CNNs ignore the **order** of inputs. Sequential data has temporal structure:
last month's price informs this month's price.

```
Dense (Ch.2):  [p1, p2, ..., p12] treated as an unordered feature vector
RNN:            processes p1 → p2 → ... → p12 one at a time, carrying h_t forward
LSTM:           adds a gated cell state c_t — a protected memory highway
```

**The key equations:**
- RNN:  $\mathbf{h}_t = \tanh(\mathbf{W}_{hh}\mathbf{h}_{t-1} + \mathbf{W}_{xh}\mathbf{x}_t + \mathbf{b}_h)$
- LSTM: $\mathbf{c}_t = \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \tilde{\mathbf{c}}_t$ (cell state update)
- GRU:  $\mathbf{h}_t = (1{-}\mathbf{z}_t) \odot \mathbf{h}_{t-1} + \mathbf{z}_t \odot \tilde{\mathbf{h}}_t$

## Running Example

The real estate platform wants a **monthly housing price index forecaster**:  
given the last 12 months of a district's median house value, predict next month's value.

We generate a **synthetic 120-month price series** (10 years):  
linear trend + annual seasonal cycle + Gaussian noise — exactly what an LSTM can decompose.

In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell


## Sliding Window Dataset

Convert the 1-D price series into `(X, y)` pairs:
- `X[i]` = months `i` through `i+T-1` (shape `(T, 1)`)
- `y[i]` = month `i+T` (the next value to predict)

**Critical:** never shuffle time-series splits. Test set must be the most recent months.

In [ ]:
# TODO: Implement this cell


## Manual RNN Forward Pass (NumPy)

$$\mathbf{h}_t = \tanh\!\left(\mathbf{W}_{hh}\mathbf{h}_{t-1} + \mathbf{W}_{xh}\mathbf{x}_t + \mathbf{b}_h\right)$$

The same weight matrices $\mathbf{W}_{hh}$ and $\mathbf{W}_{xh}$ are applied at **every** time step.

In [ ]:
# TODO: Implement this cell


## Vanishing Gradient — Visualised

How far does a gradient signal from the output reach through time?  
Each step multiplies by $\mathbf{W}_{hh}^\top$ and the Jacobian of $\tanh$.

$$\frac{\partial \mathbf{h}_T}{\partial \mathbf{h}_0} = \prod_{t=1}^{T} \mathbf{W}_{hh}^\top \cdot \mathrm{diag}(1 - \mathbf{h}_t^2)$$

When the spectral radius of $\mathbf{W}_{hh} < 1$, this product decays to zero exponentially.

In [ ]:
# TODO: Implement this cell


## LSTM — Training on the Price Series

The LSTM cell state $\mathbf{c}_t$ is updated **additively** (not multiplicatively),
so gradients can flow back 12+ steps without vanishing:

$$\mathbf{c}_t = \underbrace{\mathbf{f}_t \odot \mathbf{c}_{t-1}}_{\text{selective forget}} + \underbrace{\mathbf{i}_t \odot \tilde{\mathbf{c}}_t}_{\text{selective write}}$$

In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell


## GRU — Drop-in Comparison

The GRU replaces LSTM's three gates with two (reset + update) and eliminates the separate cell state.
Roughly 25% fewer parameters for the same hidden size.

In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell


## Hyperparameter Dial — Sequence Length $T$

How does the look-back window size affect prediction quality?
Too short: misses the seasonal cycle. Too long: more gradient propagation, slower training.

In [ ]:
# TODO: Implement this cell


## What Can Go Wrong — Demo: Shuffling Time-Series Data

Shuffling a time-series dataset before splitting leaks future prices into training.
Compare the honest (chronological) score vs the inflated (shuffled) score.

In [ ]:
# TODO: Implement this cell


## Exercises

1. **Stacked LSTM:** add a second LSTM layer (`return_sequences=True` on the first). Does it improve RMSE?
2. **Dropout:** add `layers.Dropout(0.2)` between the LSTM and Dense layers. Compare val vs train loss curves — does the gap narrow?
3. **Multi-step forecast:** instead of predicting month T+1, modify `make_windows` to predict the next 3 months (`y.shape = (N, 3)`) and add `return_sequences=True` to get one prediction per step.

In [ ]:
# TODO: Implement this cell
